# Friendly reminder

In [ ]:
import numpy as np
from sklearn.metrics import make_scorer

In [ ]:
# NumPy fundamental vectorization tricks

a = np.array([1, 2, 4])
b = np.array([2, 3, 3])
eps = 1e-13

# Conditions and masks
np.where(a > 1, a * 2, 404)
exprsn = (a > 1) * a * 2 + (a <= 1) * 404 # math equal to np.where (mask product)
np.maximum(a, b)
np.maximum(a, 0) # all negative numbers became zeros

# Defended math
np.log1p(a) # calculate log(1 + x) ~ x - x^2/2 + x^3/3, np.log(1 + x) works bad with x ~ 0
np.clip(a, eps, None) # we take ~ 0, but not 0

# Broadcasting
brdcstn = a - np.mean(a)

# Vectorizationed checking
np.any(a < 0)
np.all(b > 2)

# Aggregation
np.mean(a)
np.median(a)
np.max(a)
np.var(a) # !!! very useful

np.float64(1.5555555555555554)

In [ ]:
# How to build custom metric in scikit-learn
def custom_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# Wrap the function. Set greater_is_better to False because it's an error metric.
my_mse_scorer = make_scorer(custom_mse, greater_is_better=False)

**Принятые обозначения в регрессии**

- $y$ — вектор истинных значений (иногда обозначаются вектора полужирным через `mathbf{...}`: $\mathbf{y}$)
- $\hat{y}$ — вектор предсказаний модели
- $y_i$ — $i$-й элемент истинных данных
- $\hat{y}$ — $i$-й элемент предсказания модели
- $\bar{y}$ — среднее арифметическое истинных данных
- $n$ или $N$ — количество объектов в выборке
- $e_i$ — ошибка для $i$-го объекта

- $\text{Var}(y)$ или $\sigma^2$ — дисперсия
- $\mathbb{E}[y]$ или $\mu$ — математическое ожидание
- $L(\theta)$ или $\mathcal{L}[\theta]$ — функция правдоподобия
- $\ell(\theta)$ — логарифм правдоподобия
- $D(y, \hat{y})$ — девианс общий (для выборки)
- $d(y_i, \hat{y_i})$ — дквианс единичный (для точки)
- $\alpha$ или $q$ — квантиль (Pinball Loss)
- $p$ — индекс распределения Твинди (Power)
- $\phi$ — параметр масштаба (дисперсии) в Твиди
- $R^2$ — коэффициент детерминации
- $D^2$ — доля объяснённого девианса

# Supervised learning

## Regression

**Можно выделить 5 типов метрик регресси в `scikit-learn`**

1. **Базовые**:
  - `max_error`
  - `mean_absolute_error`
  - `mean_squared_error`
  - `median_absolute_error`
  - `root_mean_squared_error`
2. **Относительные**:
  - `mean_absolute_percentage_error`
  - `mean_squared_log_error`
  - `r2_score`
  - `root_mean_squared_log_error`
3. **Доля объяснённого**:
  - `d2_absolute_error_score`
  - `d2_pinball_score`
  - `d2_tweedie_score`
  - `explainde_variance_score`
4. **Квантильные**:
  - `mean_pinball_loss`
5. **Статистические**:
  - `mean_gamma_deviance`
  - `mean_poisson_deviance`
  - `mean_tweedie_deviance`

### 1. Базовые метрики: `MSE, RMSE, MAE, MedAE, MaxError`.

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2
$$

In [ ]:
def MSE(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2}
$$

In [ ]:
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

$$
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y_i}|
$$

In [ ]:
def MAE(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

$$
MedAE = \text{median}(|y_1 - \hat{y_1}|, ..., |y_n - \hat{y_n}|)
$$

In [ ]:
def MedAE(y_true, y_pred):
    return np.median(np.abs(y_true - y_pred))

$$
MaxError = \max_{i=\overline{1,n}}(|y_i - \hat{y_i}|)
$$

In [ ]:
def MaxError(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

### 2. Отсносительные метрики: `MAPE, R², MSLE, RMSLE`.

$$
MAPE = \frac{1}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y_i}}{y_i}\right|
$$

In [ ]:
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

$$
R^2 = 1 - \frac{MSE(y, \hat{y})}{MSE(y, \bar{y})} = 1 - \frac{ \sum_{i=1}^{n} (y_i - \hat{y_i})^2 }{ \sum_{i=1}^{n} (y_i - \bar{y_i})^2 }
$$

In [ ]:
def R2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

$$
MSLE = \frac{1}{n} \sum_{i=1}^{n} (\text{log}(y_i + 1) - \text{log}(\hat{y_i} + 1))^2
$$

In [ ]:
def MSLE(y_true, y_pred):
    return np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)

$$
RMSLE = \sqrt{MSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\text{log}(y_i + 1) - \text{log}(\hat{y_i} + 1))^2}
$$

In [ ]:
def RMSLE(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

### 3. Статистические метрики (Deviance-Based): `Poisson, Gamma, Tweedie`.

$$
MPD = \frac{1}{n} \sum_{i=1}^{n} 2 (y_i \cdot \text{log}\frac{y_i}{\hat{y_i}} - (y_i - \hat{y_i}))
$$

In [ ]:
def MPD(y_true, y_pred):
    return np.mean(2 * (y_true * np.log(y_true / y_pred) - (y_true - y_pred)))

$$
MGD = \frac{1}{n} \sum_{i=1}^{n} 2 \left(\frac{y_i}{\hat{y_i}} - \text{log}\frac{y_i}{\hat{y_i}} - 1 \right)
$$

In [ ]:
def MGD(y_true, y_pred):
    return np.mean(2 * (y_true / y_pred -  np.log(y_true / y_pred) - 1))

$$
MTD = \frac{1}{n} \sum_{i=1}^{n} 2 \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\hat{y_i}^{1-p}}{1-p} - \frac{\hat{y_i}^{2-p}}{2-p}\right),
\quad p \notin \{1, 2\}
$$

In [ ]:
def MTD(y_true, y_pred, p):
    if np.isclose(p, 1) or np.isclose(p, 2):
        raise ValueError(
            f"Параметр {p} для формулы не допустим."
    )
    y_2p = y_true ** (2 - p) / ((1 - p) * (2 - p))
    yyh_1p = y_true * y_pred ** (1 - p) / (1 - p)
    yh_2p = y_pred ** (2 - p) / (2 - p)
    return np.mean(2 * (y_2p - yyh_1p - yh_2p))

### 4. Квантильные метрики: `PinballLoss`.

$$
PinballLoss =
\begin{cases}
{
\alpha (y - \hat{y}),\quad y \ge \hat{y} \\\
(1 - \alpha) (y - \hat{y}),\quad y < \hat{y}
}
\end{cases}
$$

In [ ]:
def PL(y_true, y_pred, alpha, reduction=True):
    loss = np.maximum(alpha * (y_true - y_pred), (alpha - 1) * (y_true - y_pred))
    return np.mean(loss) if reduction else loss

### 5. Доля объяснённого: `EVS, D2AES, D2PS, D2TS`.








$$
EVS = \frac{1}{n} \sum_{i=1}^n \frac{\text{Var}(y_i - \hat{y_i})}{\text{Var}(y_i)}
$$

In [ ]:
def ELS(y_true, y_pred):
    var_sub = np.var(y_true - y_pred)
    var_tru = np.var(y_true)
    if var_true == 0:
        return 1.0 if var_sub == 0 else 0.0
    return np.mean(var_sub / var_tru)

$$
D2AES = 1 - \frac{\sum_{i=1}^{n} |y_i - \hat{y_i}|}{\sum_{i=1}^{n} | y_i - \text{median}(y)|}
$$

In [ ]:
def D2AES(y_true, y_pred):
    return 1 - np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.median(y_true)))

$$
D2PS = 1 - \frac{\sum_{i=1}^n \max(\alpha(y_i - \hat{y_i}), (\alpha - 1)(y_i - \hat{y_i}))}{\sum_{i=1}^n \max(\alpha(y_i - \bar{y_i}), (\alpha - 1)(y_i - \bar{y_i}))}
$$

In [ ]:
def D2PS(y_true, y_pred, alpha):
    numerator = np.maximum(alpha * (y_true - y_pred), (alpha - 1) * (y_true - y_pred))
    denumerator = np.maximum(alpha * (y_true - np.mean(y_true)), (alpha - 1) * (y_true - np.mean(y_true)))
    return 1 - np.sum(numerator) / np.sum(denumerator)


$$
D2TS = 1 - \frac{\sum_{i=1}^{n} \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\hat{y_i}^{1-p}}{1-p} - \frac{\hat{y_i}^{2-p}}{2-p}\right)}{ \sum_{i=1}^{n} \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\bar{y_i}^{1-p}}{1-p} - \frac{\bar{y_i}^{2-p}}{2-p}\right)}
$$

In [ ]:
def D2TS(y_true, y_pred, p):
    if np.isclose(p, 1) or np.isclose(p, 2):
        raise ValueError(
            f"Параметр {p} для формулы не допустим."
    )
    y_2p = y_true ** (2 - p) / ((1 - p) * (2 - p))
    yyh_1p = y_true * y_pred ** (1 - p) / (1 - p)
    yh_2p = y_pred ** (2 - p) / (2 - p)
    numerator = np.sum(y_2p - yyh_1p - yh_2p)
    yyb_1p = y_true * np.mean(y_true) ** (1 - p) / (1 - p)
    yb_2p = np.mean(y_true) ** (2 - p) / (2 - p)
    denumerator = np.sum(y_2p - yyb_1p - yb_2p)
    return 1 - numerator / denumerator

## Classification

**В метриках классификации, что существую в `scikit-learn` можно выделить 10 основных типов**

1. **Базовые метрики точности и ошибок**
- `accuracy_score`
- `balanced_accuracy_score`
- `top_k_accuracy_score`
- `zero_one_loss`
- `hamming_loss`

2. **Метрики на основе матрицы ошибок (precision, recall, F, Jaccard)**
- `precision_score`
- `recall_score`
- `f1_score`
- `fbeta_score`
- `jaccard_score`

3. **Матрицы ошибок**
- `confusion_matrix`
- `multilabel_confusion_matrix`
- `confusion_matrix_at_thresholds`

4. **Отчёты**
- `classification_report`
- `precision_recall_fscore_support`

5. **Вероятностные функции потерь и псевдо‑$R^2$**
- `log_loss`
- `brier_score_loss`
- `hinge_loss`
- `d2_log_loss_score`
- `d2_brier_score`

6. **Кривые и площади под кривыми (ROC, PR, DET)**
- `roc_curve`
- `roc_auc_score`
- `precision_recall_curve`
- `average_precision_score`
- `det_curve`
- `auc`

7. **Анализ по порогам**
- `metric_at_thresholds`

8. **Ранговые метрики**
- `dcg_score`
- `ndcg_score`

9. **Метрики согласия и корреляции**
- `cohen_kappa_score`
- `matthews_corrcoef`

10. **Диагностические отношения правдоподобия**
- `class_likelihood_ratios`

_Для бинарной классификации приняты стандартные обозначения:_

- _$TP$ – истинно положительные,_
- _$FP$ – ложно положительные,_
- _$TN$ – истинно отрицательные,_
- _$FN$ – ложно отрицательные._
- _$N=TP+TN+FP+FN$ – общее число объектов._

_Где $T/F$ ($True/False$) обозначает корректность соотнесения $P/N$ ($Positive/Negative$) к целевому и нецелевому классу._

### 1. Базовые метрики точности и ошибок: `accuracy_score`, `balanced_accuracy_score`, `top_k_accuracy_score`, `top_k_accuracy_score`, `zero_one_loss`, `hamming_loss`.

$$
Accuracy = \frac{TP + TN}{N}
$$

$$
BalancedAccuracy = \frac{1}{2} \left ( \frac{TP}{TP + FN} + \frac{TN}{TN + FP} \right )
$$
_для мультикласса: среднее арифметическое полнот (англ. recall) по всем классам._

_Пусть для каждого объекта модель возвращает набор $k$ наиболее вероятных классов $\hat{Y}_i^{(k)}$, тогда_
$$
Top\text{-}kAccuracy = \frac{1}{N} \sum_{i=1}^N \mathbb{1}_{ \{ y_i \in \hat{Y}_i^{(k)} \} }
$$

$$
L_{0-1} = \frac{1}{N} \sum_{i=1}^N \mathbb{1}_{ \{ y_i \neq \hat{y_i} \} }
$$

_Для мультиметочной классификации с матрицей меток $Y \in \{0, 1\}^{N \times L}$_

$$
HammingLoss = \frac{1}{N \cdot L} \sum_{i=1}^N \sum_{j=1}^L \mathbb{1}_{ [ y_{ij} \neq \hat{y_{ij}} ] }
$$

### 2. Метрики на основе матрицы ошибок: `precision_score`, `recall_score`, `f1_score`, `fbeta_score`, `jaccard_score`.

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Recall = \frac{TP}{TP + FN}
$$

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall} = 2 \cdot \frac{2TP}{2TP + FP + FN}
$$

$$
F_{\beta} = (1 + \beta^2) \frac{Precision \cdot Recall}{\beta^2 Precision + Recall} = \frac{(1 + \beta^2) TP}{(1 + \beta^2) TP + \beta^2 FN + FP}
$$

_Пусть $A$ — множество объектов класса с истинной целевой меткой, $\hat{B}$ — множество объектов предсказанных как целевой класс, тогда_
$$
Jaccard = \frac{|A \cap \hat{B}|}{|A \cup \hat{B}|} =  \frac{TP}{TP + FP + FN}
$$



### 3. Матрицы ошибок: `confusion_matrix`, `multilabel_confusion_matrix`, `confusion_matrix_at_thresholds`.

_Пусть $N$ — общее число объектов, $y_n \in \{1, \dots, K\}$ — истинный класс объекта $n$, $\hat{y_n} \in \{1, \dots, K\}$ — предсказанный класс, тогда_

$$
ConfusionMatrix = 
\begin{pmatrix}
C_{11} & \cdots & C_{1k} \\
\vdots & \ddots & \vdots \\
C_{k1} & \cdots & C_{kk}
\end{pmatrix}, \quad
C_{ij} = \sum_{n=1}^N \mathbb{1}_{\{ y_n = i,\; \hat{y}_n = j \} }.
$$

* $TP_i = C_{ii}$
* $FN_i = \sum_{j \neq i} C_{ij} = \sum_{j=1}^K C_{ij} - C_{ii}$
* $FP_i = \sum_{j \neq i} C_{ji} = \sum_{j=1}^K C_{ji} - C_{ii}$
* $TN_i = N - TP_i - FN_i - FP_i$
* $N = \sum_{i=1}^K \sum_{j=1}^K C_{ij}$

$$
MultilabelConfusionMatrix = 
\left( 
\begin{pmatrix}
TN_\ell & FP_\ell \\
FN_\ell & TP_\ell
\end{pmatrix}
\right)_{\ell=1}^{L}
$$

* $TN_\ell = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_{n,\ell}=0,\; \hat{y}_{n,\ell}=0 \} }$
* $FP_\ell = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_{n,\ell}=0,\; \hat{y}_{n,\ell}=1 \} }$
* $FN_\ell = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_{n,\ell}=1,\; \hat{y}_{n,\ell}=0 \} }$
* $TP_\ell = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_{n,\ell}=1,\; \hat{y}_{n,\ell}=1 \} }$


_Пусть $\mathcal{T} = \{t_1, t_2, \dots, t_m\}$ — множество порогов, тогда_

$$
ConfusionMatrixAtThresholds = 
\Bigl( 
\begin{pmatrix}
TN(t) & FP(t) \\
FN(t) & TP(t)
\end{pmatrix}
\Bigr)_{t \in \mathcal{T}}
$$

* $TP(t) = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_n = 1,\; \hat{y}_n(t) = 1 \} }$
* $TN(t) = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_n = 0,\; \hat{y}_n(t) = 0 \} }$
* $FP(t) = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_n = 0,\; \hat{y}_n(t) = 1 \} }$
* $FN(t) = \sum_{n=1}^{N} \mathbf{1}_{ \{ y_n = 1,\; \hat{y}_n(t) = 0 \} }$

_предсказание при пороге $t$ определяется как $\hat{y}_n(t) = \mathbb{1}_{ \{ p_n \ge t \} }$,
где $p_n \in [0,1]$ — вероятность положительного класса для объекта $n$._


### 4. Отчёты: `classification_reposrt`, `precision_recall_fscore_support`.

$$
\begin{aligned}
TP_k &= \sum_{i=1}^N \mathbb{1}_{ \{ y_i = k,\; \hat{y}_i = k \} }, &
FP_k &= \sum_{i=1}^N \mathbb{1}_{ \{ y_i \neq k,\; \hat{y}_i = k \} }, \\
FN_k &= \sum_{i=1}^N \mathbb{1}_{ \{ y_i = k,\; \hat{y}_i \neq k \} }, &
TN_k &= \sum_{i=1}^N \mathbf{1}_{ \{ y_i \neq k,\; \hat{y}_i \neq k \} }.
\end{aligned}
$$

$$
\text{support}_k = \sum_{i=1}^N \mathbb{1}_{ \{ y_i = k \} } = TP_k + FN_k.
$$
_— число реальных объектов этого класса._

_для каждой метрики для каждого класса ($precision_k$, $recall_k$, $F_{\beta, k}$), если знаменатель равен нулю, то метрика равна 0._

$$
ClassificationReport =
\begin{array}{c|cccc}
Class & precision & recall & f1-score & support \\
\hline
1 & p_1 & r_1 & f_1 & s_1 \\
\vdots & \vdots & \vdots & \vdots & \vdots \\
K & p_K & r_K & f_K & s_K \\
\hline
macro\space avg & \bar{p}_{macro} & \bar{r}_{macro} & \bar{f}_{macro} & \sum s_k = N \\
micro\space avg & \bar{p}_{micro} & \bar{r}_{micro} & \bar{f}_{micro} & N 
\\
weighted\space avg & \bar{p}_{weighted} & \bar{r}_{weighted} & \bar{f}_{weighted} & N
\end{array}
$$

* **Macro-average** (среднее по классам без учёта поддержки):
  $$
  \bar{p}_{macro} = \frac{1}{K} \sum_{k=1}^K precision_k, \qquad
  \bar{r}_{macro} = \frac{1}{K} \sum_{k=1}^K recall_k, \qquad
  \bar{f}_{macro} = \frac{1}{K} \sum_{k=1}^K F_{1,k}.
  $$

* **Micro-average** (глобальное агрегирование через суммирование $TP/FP/FN$):
  $$
  \bar{p}_{micro} = \frac{\sum_k TP_k}{\sum_k TP_k + \sum_k FP_k}, \qquad
  \bar{r}_{micro} = \frac{\sum_k TP_k}{\sum_k TP_k + \sum_k FN_k}, \qquad
  \bar{f}_{micro} = \frac{2 \cdot \bar{p}_{micro} \cdot \bar{r}_{micro}}{\bar{p}_{micro} + \bar{r}_{micro}}.
  $$

* **Weighted-average** (среднее, взвешенное по поддержке классов):
  $$
  \bar{p}_{weighted} = \frac{\sum_k support_k \cdot precision_k}{N}, \qquad
  \bar{r}_{weighted} = \frac{\sum_k support_k \cdot recall_k}{N}, \qquad
  \bar{f}_{weighted} = \frac{\sum_k support_k \cdot F_{1,k}}{N}.
  $$

$$
PrecisionRecallFscoreSupport = 
\bigl( 
precision, \; recall, \; fscore, \; support
\bigr)
\in (\mathbb{R}^K)^4
$$

### 5. Вероятностные функции потерь и пвсевдо-$R^2$: `log_loss`, `brier_score_loss`, `hinge_loss`, `d2_log_loss_score`, `d2_brier_score`.

$$
LogLoss = - \frac{1}{N} \sum_{i=1}^N [y_i \log p_i + (1 - y_i) \log (1 - p_i)]
$$

$$
BrierScoreLoss = \frac{1}{N} \sum_{i=1}^N (p_i - y_i)^2
$$

$$
HingeLoss = \frac{1}{N} \sum_{i=1}^N max(0, 1-y_i\cdot \hat{y_i})
$$

$$
D2LogLossScore = 1 - \frac{LogLoss_{model}}{LogLoss_{null}}
$$

$$
D2BrierScore = 1 - \frac{BrierScoreLoss_{model}}{BrierScoreLoss_{null}}
$$

### 6. Кривые и площади под кривыми: `roc_curve`, `roc_auc_score`, `precision_recall_curve`, `average_precision_score`, `det_curve`, `auc`.

$Curve = \bigl ( X\_vec(t),\; Y\_vec(t),\; thresholds \bigr), \quad \text{где } t \in thresholds$

_обобщённый формат выходных данных всех функций кривых_

$$
RocCurve = 
\Bigl \{ 
\bigl(
FPR(t),\; TPR(t)
\bigr) 
\; \Big| \;
t \in \mathcal{T}
\Bigr \}
$$

_где $\mathcal{T} = \{t_1, \dots, t_m\}$ — множество всех расматриваемых порогов._

* $TPR(t) = \frac{TP(t)}{TP(t) + FN(t)}$
* $FPR(t) = \frac{FP(t)}{FP(t) + TN(t)}$

$$
RocAucScore = \int_\mathcal{T} TPR(t)\; d(FPR(t)) = \sum_{i=1}^{m-1} \frac{TPR(t_i) + TPR(t_{i+1})}{2} \cdot \bigl| FPR(t_i) - FPR(t_{i+1}) \bigr|
$$
_эмпирически равна вероятности того, что случайно выбранный положительный объект получит score выше, чем случайно выбранный отрицательный._

$$
PrecisionRecallCurve = 
\Bigl \{ 
\bigl(
Precision(t),\; Recall(t)
\bigr) 
\; \Big| \;
t \in \mathcal{T}
\Bigr \}
$$

$$
AveragePrecisionScore = \int_\mathcal{T} Precision(t) \; d(Recall(t)) = \sum_{n} (Recall_n - Recall_{n-1}) \cdot Precision_n
$$

$$
DETCurve = 
\Bigl \{
\bigl (
FPR(t), \; FNR(t)
\bigr )
\; \Big| \;
t \in \mathcal(T)
\Bigr \}
$$

* $FPR(t) = \frac{FP(t)}{FP(t) + TN(t)}$
* $FNR(t) = \frac{FN(t)}{TP(t) + FN(t)} = 1 - TPR(t)$

_DET — Detection Error Tradeoff_

$$
AUC = \sum_{k=1}^{n-1} \frac{(x_{k+1} - x_k)(y_{k+1} + y_k)}{2} 
$$

### 7. Анализ по порогам: `metric_at_thresholds`.

$$
MetricAtThresholds = 
\Bigl \{
\bigl (
t, \; M(CM(t))
\; \Big| \;
t \in \mathcal{T}
\bigr )
\Bigr \}
$$

_где_

$$
CM(t) =
\begin{pmatrix}
TN(t) & FP(t)\\
FN(t) & TP(t)
\end{pmatrix}
$$
$$
\hat{y}_n(t) = \mathbb{1}_{\{p_n \ge t\}}
$$

### 8. Ранговые метрики: `dcg_score`, `ndcg_score`.

_Пусть $y \in \mathbb{R}^N$ — истинные релевантности, $\hat{s} \in \mathbb{R}^N$ — предсказанные скоры. $\pi$ — перестановка, сортирующая объекты по убыванию $\hat{s}$._

_Функция усиления: $g(y) = y$ (linear) или $g(y) = 2^y - 1$ (exponential, по умолчанию)._

$$
DCGScore = \sum_{i=1}^N \frac{g(y_{\pi(i)})}{\log_2(i+1)}
$$

$$
NDCGScore = \frac{DCG}{IDCG}
$$
$$
IDCG = \sum_{i=1}^N \frac{g(y_{\pi^*(i)})}{\log_2(i+1)}
$$

_если $IDCGk = 0$, то $NDCGk = 0$_

_$\pi^*$ — идеальная перестановка по убыванию истинной релевантности $y$._

### 9. Метрики cсогласия и корреляции: `cohen_kappa_score`, `matthews_corrcoef`.

$$
CohenKappaScore = \frac{p_o - p_e}{1 - p_e}
$$

$$
p_o = \frac{1}{N} \sum_{i=1}^N \mathbb{1}_{\{y_i = \hat{y}_i\}} = \frac{1}{N}\sum_k C_{kk}
$$

$$
p_e = 
\sum_{k=1}^K 
\underbrace{\left(
\frac{1}{N}\sum_i \mathbb{1}_{\{y_i=k\}}
\right)
}_{\text{доля истин класса }k} 
\cdot 
\underbrace{\left(
\frac{1}{N}\sum_i \mathbb{1}_{\{\hat{y}_i=k\}}
\right)
}_{\text{доля предсказаний класса }k} =
\frac{1}{N^2} \sum_{k=1}^K 
\Bigl [
\sum_i \mathbb{1}_{\{y_i=k\}} \cdot \sum_i \mathbb{1}_{\{\hat{y}_i=k\}}
\Bigr ]
$$

$$
MatthewsCorrcoef = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}
$$

_мультикласс_

$$
MatthewsCorrcoef = \frac{c \cdot s - \sum_{k=1}^K p_k t_k}{\sqrt{ (s^2 - \sum_k p_k^2)(s^2 - \sum_k t_k^2) }}
$$

_при знаменателе 0 возвращается 0._
* $C_{ij}$ — число объектов класса $i$, предсказанных как $j$
* $p_k = \sum_j C_{kj}$ (истинные)
* $t_k = \sum_i C_{ik}$ (предсказанные)
* $c = \sum_k C_{kk}$
* $s = \sum_{i,j} C_{ij} = N$

### 10. Диагностические оношения правдоподобия: `class_likelihood_ratios`.

$$
\text{ClassLikelihoodRatios} = \bigl( \mathbf{LR_+},\; \mathbf{LR_-} \bigr)
$$

_где_

$$
\mathbf{LR_+} = \left( LR_+^{(1)},\; LR_+^{(2)},\; \dots,\; LR_+^{(K)} \right), \qquad
\mathbf{LR_-} = \left( LR_-^{(1)},\; LR_-^{(2)},\; \dots,\; LR_-^{(K)} \right),
$$

_а для каждого класса $k$ (по схеме one‑vs‑rest)_ 

$$
LR_+^{(k)} = \frac{TPR_k}{FPR_k} = \frac{TP_k / (TP_k + FN_k)}{FP_k / (FP_k + TN_k)}, \\
LR_-^{(k)} = \frac{FNR_k}{TNR_k} = \frac{FN_k / (TP_k + FN_k)}{TN_k / (FP_k + TN_k)}.
$$

_при знаменателе 0 возвращается $\infty$ или `np.nan`._

# Unsupervised learning

## Clasterization

**Метрики кластеризации в `scikit-learn` можно разбить на логические группы по характеру оценки и необходимым входным данным.**

1. **Внутренние метрики (только разметка кластеров, без эталонных меток)**
- `calinski_harabasz_score`
- `davies_bouldin_score`
- `silhouette_score`
- `silhouette_samples`

2. **Матрицы для сравнения двух кластеризаций**
- `cluster.contingency_matrix`
- `cluster.pair_confusion_matrix`

3. **Метрики на основе подсчёта пар объектов (Rand-семейство и Fowlkes-Mallows)**
- `rand_score`
- `adjusted_rand_score`
- `fowlkes_mallows_score`

4. **Метрики на основе взаимной информации и энтропии**
- `mutual_info_score`
- `adjusted_mutual_info_score`
- `normalized_mutual_info_score`
- `homogeneity_score`
- `completeness_score`
- `v_measure_score`
- `homogeneity_completeness_v_measure` (возвращает сразу homogeneity, completeness и V-measure)

### 1. Внутренние метрики: `calinski_harabasz_score`, `davies_bouldin_score`, `silhouette_score`, `silhouette_samples`.

$$
CalinskiHarabaszScore = \frac{\mathrm{tr}(B_k)}{\mathrm{tr}(W_k)} \cdot \frac{n - k}{k - 1}
$$

где $W_k = \sum_{q=1}^{k}\sum_{x \in C_q}(x - c_q)(x - c_q)^T$, $\quad B_k = \sum_{q=1}^{k} n_q (c_q - c)(c_q - c)^T$, $n$ — число объектов, $k$ — число кластеров.

$$
DaviesBouldinScore = \frac{1}{k}\sum_{i=1}^{k} \max_{j \neq i} \left( \frac{s_i + s_j}{d_{ij}} \right)
$$

_где $s_i$ — средний разброс внутри кластера $i$, $d_{ij}$ — расстояние между центроидами кластеров $i$ и $j$._


$$
SilhouetteScore = \frac{1}{n}\sum_{i=1}^{n} s(i), \qquad s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}
$$

_где $a(i)$ — среднее расстояние до объектов своего кластера, $b(i)$ — минимальное среднее расстояние до объектов другого кластера._

$$
SilhouetteSamples = s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}, \quad i = 1,\dots,n
$$

_(возвращает вектор силуэтных коэффициентов для каждого объекта)_

### 2. Матрицы для сравнения двух кластеризаций: `cluster.contingency_matrix`, `cluster.pair_confusion_matrix`.

$$
ClusterContingencyMatrix = \Bigl ( C_{ij} \Bigr )_{i,} \quad C_{ij} = \big|\, \{\, x : x \in U_i \wedge x \in V_j \,\} \,\big|
$$

_где $U_i$ — $i$-й класс истинной разметки, $V_j$ — $j$-й кластер предсказанной разметки._

$$
ClusterPairConfusionMatrix = 
\begin{pmatrix}
C_{00} & C_{01} \\
C_{10} & C_{11}
\end{pmatrix}
$$

$$
C_{11} = \sum_{ij} \binom{n_{ij}}{2}, \quad
C_{10} = \sum_{i}\binom{a_i}{2} - C_{11}, \quad
C_{01} = \sum_{j}\binom{b_j}{2} - C_{11}, \quad
C_{00} = \binom{n}{2} - C_{11} - C_{10} - C_{01}
$$

_где элементы отражают пары объектов, согласованные/несогласованные между двумя разбиениями._

### 3. Метрики на основе подсчёта пар объектов: `rand_score`, `adjusted_rand_score`, `fowlkes_mallows_score`.

_пусть_
- _$a$ — число пар в одном кластере в обоих разбиениях,_
- _$b$ — число пар в разных кластерах в обоих разбиениях,_
- _$c, d$ — число пар, согласованных только в одном из разбиений._

$$
RandScore = \frac{a + b}{a + b + c + d} = \frac{a + b}{\binom{n}{2}}
$$

$$
AdjustedRandScore 
= 
\frac{\mathrm{RI} - \mathbb{E}[\mathrm{RI}]}{\max(\mathrm{RI}) - \mathbb{E}[\mathrm{RI}]}
=
\frac{\sum_{ij}\binom{n_{ij}}{2} - \left[\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}\right] / \binom{n}{2}}{\tfrac{1}{2}\left[\sum_i\binom{a_i}{2} + \sum_j\binom{b_j}{2}\right] - \left[\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}\right] / \binom{n}{2}}
$$

$$
\mathrm{RI} = \frac{\sum_{ij} \binom{n_{ij}}{2}}{\binom{N}{2}}
$$

_где $n_{ij}$ – число объектов в пересечении кластера $i$ истинной разметки и кластера $j$ предсказанной._

_(Rand Index) – доля пар объектов, которые классифицируются **одинаково** (т.е. либо обе пары в одном кластере в обеих разбивках, либо обе в разных)._

### 4. Метрики на основе взаимной информации и энтропии: `mutual_info_score`, `adjusted_mutual_info_score`, `normalized_mutual_info_score`, `homogeneity_score`, `completeness_score`, `v_measure_score`, `homogeneity_completeness_v_measure`.

_пусть $P(i) = a_i/n$, $P'(j) = b_j/n$, $P(i,j) = n_{ij}/n$_

$$
MutualInfoScore = \sum_{i=1}^{|U|}\sum_{j=1}^{|V|} \frac{n_{ij}}{n} \log\!\left( \frac{n \, n_{ij}}{a_i \, b_j} \right) \quad (\mathrm{MI})
$$

$$
AdjustedMutualInfoScore = \frac{\mathrm{MI}(U,V) - \mathbb{E}[\mathrm{MI}(U,V)]}{\mathrm{mean}\big(H(U),\, H(V)\big) - \mathbb{E}[\mathrm{MI}(U,V)]}
$$

$$
NormalizedMutualInfoScore = \frac{\mathrm{MI}(U,V)}{\mathrm{mean}\big(H(U),\, H(V)\big)}
$$

_(по умолчанию усреднение — среднее арифметическое энтропий)_

$$
HomogeneityScore = 1 - \frac{H(C \mid K)}{H(C)}
$$

_где $C$ — истинные классы, $K$ — кластеры; равно $1$, если каждый кластер содержит объекты только одного класса._

$$
CompletenessScore = 1 - \frac{H(K \mid C)}{H(K)}
$$

_равно $1$, если все объекты одного класса попадают в один кластер._

$$
VMeasureScore = \frac{(1 + \beta) \cdot h \cdot c}{\beta \cdot h + c}, \qquad \beta = 1 \;\Rightarrow\; VMeasureScore = \frac{2 \cdot h \cdot c}{h + c}
$$

_где $h$ — homogeneity, $c$ — completeness (гармоническое среднее при $\beta=1$)._

$$
HomogeneityCompletenessVMeasure = \big(\, h,\; c,\; v \,\big)
$$

$$
h = 1 - \frac{H(C\mid K)}{H(C)}, \qquad
c = 1 - \frac{H(K\mid C)}{H(K)}, \qquad
v = \frac{2 h c}{h + c}
$$
